# Local Maximum and Minimum Surface Temperature
 
```{glue:figure} trend_fig_max_min
:scale: 50%
:align: right
```
**Figure. Annual maximum (red) and minimum (blue) temperature at Koror.** The solid black line represents a trend that is statistically significant (p < 0.05).  The dashed black line represents a trend that is not statistically significant.  

## Setup

First, we need to import all the necessary libraries. Some of them are specifically developed to handle the download and plotting of the data and are hosted at the [indicators set-up repository](https://github.com/lauracagigal/indicators_setup) in GitHub

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os.path as op
import sys
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "notebook_connected"

from myst_nb import glue 

sys.path.append("../../../../indicators_setup")
from ind_setup.plotting_int import plot_timeseries_interactive, fig_int_to_glue

sys.path.append("../../functions")
from data_downloaders import GHCN
from air_temp import load_site_config

## Define location and variables of interest

In [3]:
site_name = "Palau"

site_config_path = Path(f'../../data/sites/{site_name}.json')
site_cfg = load_site_config(site_config_path)

site_name = site_cfg.get('site_name', 'Site')
country = site_cfg['country']
ghcn_station_id = site_cfg['ghcn_station_id']
ghcn_station_name = site_cfg.get('ghcn_station_name', '')
vars_interest = list(site_cfg.get('vars_interest', ['TMIN', 'TMAX']))

## Get Data

In [4]:
data_base_dir = Path('../../data')
path_figs = "../../matrix_cc/figures"
data_dir = Path(data_base_dir, 'air_temp')
output_dir = data_base_dir

output_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

path_data = str(data_dir)

## Observations from Koror Station

https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/doc/GHCND_documentation.pdf

The data used for this analysis comes from the GHCN (Global Historical Climatology Network)-Daily database. <br>
This a database that addresses the critical need for historical daily temperature, precipitation, and snow records over global land areas. GHCN-Daily is a
composite of climate records from numerous sources that were merged and then subjected to a suite of
quality assurance reviews. The archive includes over 40 meteorological elements including temperature daily maximum/minimum, temperature at observation time,
precipitation and more.

In [5]:
print(f"Site: {site_name} ({country}) - GHCN station {ghcn_station_id} {ghcn_station_name}")

Site: Palau (Palau) - GHCN station PSW00040309 PW KOROR GSN 91408


In [6]:
pickle_path = op.join(path_data, f'GHCN_{ghcn_station_id}.pkl')
st_data = pd.read_pickle(pickle_path)
print(f'Loaded {len(st_data)} daily records from {pickle_path}')

Loaded 25966 daily records from ../../data/air_temp/GHCN_PSW00040309.pkl


In [7]:
st_data_daily = st_data.copy()

In [8]:
print(st_data_daily.TMIN.mean(), st_data_daily.TMAX.mean())

24.39285989370715 31.05149041053685


In [9]:
dict_plot = [{'data' : st_data_daily, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'}]
fig = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12))


In [10]:
dict_plot = [{'data' : st_data_daily, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'}]
fig = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12))


In [11]:
st_data = st_data.resample('YE').mean()
glue("n_years", len(np.unique(st_data.index.year)), display=False)
glue("start_year", st_data.dropna().index[0].year, display=False)
glue("end_year", st_data.dropna().index[-1].year, display=False)

## Plotting Annual

At this piece of code we will plot the Mean annual temperature over time and its associated trend

The following plot represent the average minimum and maximum temperature over time

### Minimum Temperatures

In [12]:
dict_plot = [{'data' : st_data, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'},
        ]

In [13]:
fig, trend_minimum = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(op.join(path_figs, 'F3_ST_min.html'), include_plotlyjs="cdn")

### Maximum Temperatures

In [14]:
dict_plot = [{'data' : st_data, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'},
        ]

In [15]:
fig, trend_maximum = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(op.join(path_figs, 'F3_ST_max.html'), include_plotlyjs="cdn")

In [16]:
print(st_data.TMIN.mean(), st_data.TMAX.mean())

24.4096970843879 31.033677983557787


### Minimum and Maximum Temperatures

In [17]:
dict_plot = [{'data' : st_data, 'var' : 'TMIN', 'ax' : 1, 'label' : 'TMIN'},
        {'data' : st_data, 'var' : 'TMAX', 'ax' : 1, 'label' : 'TMAX'},
        # {'data' : st_data, 'var' : 'diff', 'ax' : 1, 'label' : 'Difference TMAX - TMIN'}
        ]

In [18]:
fig, TRENDS = plot_timeseries_interactive(dict_plot, label_yaxes = 'Temperature [ºC]', trendline=True, figsize = (24, 11), return_trend = True)
fig.write_html(op.join(path_figs, 'F3_ST_min_max.html'), include_plotlyjs="cdn")
fig.write_image(op.join(path_figs, 'F3_ST_min_max.png'), width=1200, height=600,)

glue("trend_min", float(TRENDS[0]), display=False)
glue("trend_max", float(TRENDS[1]), display=False)

glue("change_min", float(TRENDS[0]*len(np.unique(st_data.index.year))), display=False)
glue("change_max", float(TRENDS[1]*len(np.unique(st_data.index.year))), display=False)

glue("trend_fig_max_min", fig_int_to_glue(fig), display=False)

**Fig.** Annual maximum (red) and minimum (blue) temperature at Koror. The solid black line represents a trend that is statistically significant (p < 0.05).  The dashed black line represents a trend that is not statistically significant.   

## Difference temperature

The following plot represents the avergae difference between the minimum and maximum temperature over time. The decreasing trend means that the daily variability is decreasing over time

In [19]:
dict_plot = [{'data' : st_data, 'var' : 'diff', 'ax' : 1, 'label' : 'Difference TMAX - TMIN'}]
fig, trend = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True)
glue("trend_diff", float(trend[0]), display=False)


**Fig.** Annual maximum of the difference of the maximum and minimum temperature within each day

### Generate table
The final step is to generate a table sumarizing different metrics of the data analyzed in the plots above

In [20]:
from ind_setup.tables import table_temp_12, style_matrix

style_matrix(table_temp_12(st_data, st_data_daily, trend_maximum[0], trend_minimum[0]))

Metric,Value
Annual Maximum Temperature (°C),32.104
Change in Annual Maximum Temperature since 1951,0.150
Rate of Change in Annual Maximum Temperature (°C/year),0.002
Annual Minimum Temperature (°C),23.757
Change in Annual Minimum Temperature since 1951,1.125
Rate of Change in Annual Minimum Temperature (°C/year),0.015
Mean Daily Mean Temperature (°C),27.722
Mean Daily Maximum Temperature (°C),31.051
Mean Daily Minimum Temperature (°C),24.393
